# Importador histórico unificado MER v2 — cualquier mes

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Importador_Historicos_Unificado_MER_v2.ipynb)

Configura `ANIO`, `MES` y los nombres reales de los tres ZIP. Prepara las seis entidades y un manifiesto con período, fuentes y totales. **No escribe en Google Sheets.**

In [ ]:
!git clone -q https://github.com/juliopez/MercadoPublico-Inteligencia.git /content/MercadoPublico-Inteligencia
from google.colab import files
print('Sube los tres ZIP oficiales del mismo mes, uno tras otro')
subidos=files.upload()
print('Recibidos:',list(subidos))

In [ ]:
import pathlib,subprocess,sys,re
ANIO=2026
MES=10
# Nombres configurables, sin renombrar archivos ZIP oficiales.
nombres={'licitaciones':f'{ANIO}-{MES}.zip',
         'cot':f'COT_{ANIO}-{MES:02d}.zip',
         'oc':f'{ANIO}-{MES} (OC).zip'}
for tipo,nombre in nombres.items():
 assert pathlib.Path(nombre).is_file(),f'Falta {tipo}: {nombre}'
periodo=f'{ANIO}-{MES:02d}'
out=f'/content/preparado_mercado_publico_{periodo}'
cmd=[sys.executable,'/content/MercadoPublico-Inteligencia/scripts/importador_historicos_v2.py',
 '--anio',str(ANIO),'--mes',str(MES),
 '--licitaciones',nombres['licitaciones'],'--cot',nombres['cot'],'--oc',nombres['oc'],
 '--out',out]
subprocess.run(cmd,check=True)
print('Preparación terminada',periodo,'sin modificar Sheets')

In [ ]:
from google.colab import files
import json,zipfile,pathlib
carpeta=pathlib.Path(out)
print(json.dumps(json.loads((carpeta/'informe.json').read_text()),indent=2,ensure_ascii=False))
salida=f'/content/preparado_mercado_publico_mer_v2_{periodo}.zip'
with zipfile.ZipFile(salida,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in carpeta.iterdir():
  if f.suffix in ('.csv','.json'):z.write(f,f.name)
files.download(salida)
print('Se generó el paquete para',periodo,'. No se ha escrito en Sheets.')